In [14]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as scipy_stats
import gpboost as gpb
import patsy
import re

In [15]:
results_path = 'results_0'
formats = ['parquet', 'delta', 'hudi', 'iceberg']
data_sets = ['tpcds_1', 'tpcds_10', 'tpcds_100']
optimizations = ['none', 'zorder', 'bloom', 'partitioning']
block_sizes = ['64MiB', '128MiB', '256MiB']
data_frames = []

In [16]:
for format in formats:
    for data_set in data_sets:
        for optimization in optimizations:
            for block_size in block_sizes:
                if format == 'parquet' and (optimization != 'none' or block_size != '128MiB'):
                    continue
                
                file_path = f'{results_path}/{format}/{data_set}/{optimization}/{block_size}/results.csv'
                df = pd.read_csv(file_path)
                df['format'] = format
                df['data_set'] = data_set
                df['optimization'] = optimization
                df['block_size'] = block_size
                data_frames.append(df)

In [17]:
master_df = pd.concat(data_frames, ignore_index=True)
target_metric = 'elapsedTime'
master_df[target_metric] = master_df[target_metric].round(2)
summary_stats_by_query = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'query'])[target_metric].agg(
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max',
    cv=lambda x: np.std(x, ddof=1) / np.mean(x) * 100 # coef of variation
).reset_index()
display(summary_stats_by_query)

,format,data_set,optimization,block_size,query,mean,median,std,min,max,cv
0,delta,tpcds_1,bloom,128MiB,q1,694.2,438.5,805.629885,430,2987,116.051554
1,delta,tpcds_1,bloom,128MiB,q10,1200.3,1190.5,39.477279,1138,1272,3.288951
2,delta,tpcds_1,bloom,128MiB,q10a,1148.6,1113.0,63.328421,1095,1260,5.513531
3,delta,tpcds_1,bloom,128MiB,q11,917.8,905.5,76.179321,869,1128,8.300209
4,delta,tpcds_1,bloom,128MiB,q12,96.3,92.0,10.698390,88,124,11.109440
...,...,...,...,...,...,...,...,...,...,...,...
13093,parquet,tpcds_100,none,128MiB,q95,22784.9,22728.0,501.340525,22001,23567,2.200319
13094,parquet,tpcds_100,none,128MiB,q96,1099.5,1084.5,59.920781,1013,1199,5.449821
13095,parquet,tpcds_100,none,128MiB,q97,11490.2,11499.0,75.596590,11346,11599,0.657922
13096,parquet,tpcds_100,none,128MiB,q98,2101.2,1993.5,345.296266,1847,3047,16.433289


In [18]:
print("--- Shapiro-Wilk Test for Normality ---")
groups_shapiro = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'query'])[target_metric].apply(list)

df_shapiro_list = []

for name, group_data in groups_shapiro.items():
    stat, p_val = scipy_stats.shapiro(group_data)

    df_shapiro_list.append({
        'Group': name,
        'W_Statistic': stat,
        'p_value': p_val.round(4),
        'Distribution': 'Normal' if p_val > 0.05 else 'Not Normal'
    })

df_shapiro = pd.DataFrame(df_shapiro_list)
display(df_shapiro)

df_shapiro_normal = df_shapiro[df_shapiro['Distribution'] == 'Normal']
display(df_shapiro_normal)

df_shapiro_not_normal = df_shapiro[df_shapiro['Distribution'] == 'Not Normal']
display(df_shapiro_not_normal)

--- Shapiro-Wilk Test for Normality ---


/home/heavylight/ResultsAnalysis/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


,Group,W_Statistic,p_value,Distribution
0,"(delta, tpcds_1, bloom, 128MiB, q1)",0.372588,0.0000,Not Normal
1,"(delta, tpcds_1, bloom, 128MiB, q10)",0.934044,0.4888,Normal
2,"(delta, tpcds_1, bloom, 128MiB, q10a)",0.804009,0.0162,Not Normal
3,"(delta, tpcds_1, bloom, 128MiB, q11)",0.590802,0.0000,Not Normal
4,"(delta, tpcds_1, bloom, 128MiB, q12)",0.721175,0.0016,Not Normal
...,...,...,...,...
13093,"(parquet, tpcds_100, none, 128MiB, q95)",0.961838,0.8066,Normal
13094,"(parquet, tpcds_100, none, 128MiB, q96)",0.952258,0.6953,Normal
13095,"(parquet, tpcds_100, none, 128MiB, q97)",0.938044,0.5315,Normal
13096,"(parquet, tpcds_100, none, 128MiB, q98)",0.626746,0.0001,Not Normal


,Group,W_Statistic,p_value,Distribution
1,"(delta, tpcds_1, bloom, 128MiB, q10)",0.934044,0.4888,Normal
6,"(delta, tpcds_1, bloom, 128MiB, q14)",0.938551,0.5370,Normal
8,"(delta, tpcds_1, bloom, 128MiB, q14b)",0.862004,0.0806,Normal
9,"(delta, tpcds_1, bloom, 128MiB, q15)",0.921950,0.3735,Normal
11,"(delta, tpcds_1, bloom, 128MiB, q17)",0.943248,0.5897,Normal
...,...,...,...,...
13091,"(parquet, tpcds_100, none, 128MiB, q93)",0.907208,0.2624,Normal
13093,"(parquet, tpcds_100, none, 128MiB, q95)",0.961838,0.8066,Normal
13094,"(parquet, tpcds_100, none, 128MiB, q96)",0.952258,0.6953,Normal
13095,"(parquet, tpcds_100, none, 128MiB, q97)",0.938044,0.5315,Normal


,Group,W_Statistic,p_value,Distribution
0,"(delta, tpcds_1, bloom, 128MiB, q1)",0.372588,0.0000,Not Normal
2,"(delta, tpcds_1, bloom, 128MiB, q10a)",0.804009,0.0162,Not Normal
3,"(delta, tpcds_1, bloom, 128MiB, q11)",0.590802,0.0000,Not Normal
4,"(delta, tpcds_1, bloom, 128MiB, q12)",0.721175,0.0016,Not Normal
5,"(delta, tpcds_1, bloom, 128MiB, q13)",0.447797,0.0000,Not Normal
...,...,...,...,...
13060,"(parquet, tpcds_100, none, 128MiB, q69)",0.797005,0.0133,Not Normal
13063,"(parquet, tpcds_100, none, 128MiB, q70a)",0.545676,0.0000,Not Normal
13074,"(parquet, tpcds_100, none, 128MiB, q8)",0.742627,0.0029,Not Normal
13092,"(parquet, tpcds_100, none, 128MiB, q94)",0.555992,0.0000,Not Normal


In [19]:
master_df = pd.concat(data_frames, ignore_index=True)
summary_stats_aggregated_runs = master_df.groupby(['format', 'data_set', 'optimization', 'block_size', 'run_id'])[['elapsedTime']].sum().reset_index()
display(summary_stats_aggregated_runs)

,format,data_set,optimization,block_size,run_id,elapsedTime
0,delta,tpcds_1,bloom,128MiB,0,102304
1,delta,tpcds_1,bloom,128MiB,1,89034
2,delta,tpcds_1,bloom,128MiB,2,90110
3,delta,tpcds_1,bloom,128MiB,3,88166
4,delta,tpcds_1,bloom,128MiB,4,87548
...,...,...,...,...,...,...
1105,parquet,tpcds_100,none,128MiB,5,1698289
1106,parquet,tpcds_100,none,128MiB,6,1755832
1107,parquet,tpcds_100,none,128MiB,7,1827483
1108,parquet,tpcds_100,none,128MiB,8,1722880


In [20]:
target_metrics = ['elapsedTime', 'executorCpuTime', 'executorRunTime', 'resultSize', 'peakExecutionMemory', 'shuffleTotalBytesRead', 'shuffleBytesWritten']
EPSILON = 1e-6

In [21]:
def print_df_latex(df, caption):
    latex_output = (
        df.style
        .hide(axis='index')
        .to_latex(
            caption=caption, 
            position='h!', 
            position_float='centering',
            hrules=True
        )
    )
    lines = latex_output.split('\n')
    caption_line = next((l for l in lines if l.strip().startswith('\\caption{')), None)
    
    if caption_line:
        lines.remove(caption_line)
        # Find the index of \end{table} and insert the caption before it
        end_table_idx = next(i for i, l in enumerate(lines) if l.strip() == '\\end{table}')
        lines.insert(end_table_idx, caption_line)
        latex_output = '\n'.join(lines)

    print(latex_output)

def clean_patsy_name(name):
    if name == 'Intercept':
        return name
    cleaned = re.sub(r'C\([^,)]+,\s*Treatment\([^)]+\)\)', '', name)
    cleaned = re.sub(r'\[T\.([^\]]+)\]', r' \1', cleaned)
    return cleaned.title().strip()

In [22]:
for target_metric in target_metrics:
        
    for data_set in data_sets:

        df_rq1 = master_df[
            (master_df['optimization'] == 'none') &
            (master_df['block_size'] == '128MiB') &
            (master_df['data_set'] == data_set) 
        ].copy()

        df_rq1['format'] = pd.Categorical(df_rq1['format'], categories=formats)

        y = df_rq1[target_metric].values + EPSILON
        group_data = df_rq1['query'].to_numpy()

        X_df = patsy.dmatrix('format', data=df_rq1, return_type='dataframe')
        X = X_df.values
        feature_names = X_df.columns.tolist()

        gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
        gp_model.fit(y=y, X=X)
        fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

        summary_data = []
        for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
            z_stat = coef / std_err
            p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
            pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
            
            summary_data.append({
                'Factor': 'Format Parquet' if name == 'Intercept' else clean_patsy_name(name),
                'Coefficient $\\beta$': f'{coef:.4f}',
                'Std. Error': std_err  if name != 'Intercept' else '',
                'Percentage Change': f'{pct_change:+.1f}\\%',
                '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
            })

        summary_df = pd.DataFrame(summary_data)
        caption_text = f"TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}"
        print_df_latex(summary_df, caption_text)

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Factor & Coefficient $\beta$ & Std. Error & Percentage Change & $p$-Value & $H_0$ Rejected \\
\midrule
Format Parquet & 5.1505 &  & +0.0\% &  &  \\
Format Delta & 1.3351 & 0.019385 & +280.0\% & 0.0000 & True \\
Format Hudi & 0.1607 & 0.017432 & +17.4\% & 0.0000 & True \\
Format Iceberg & 0.0257 & nan & +2.6\% & nan & False \\
\bottomrule
\end{tabular}
\caption{TPC-DS 1GB, Elapsed Time}
\end{table}

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Factor & Coefficient $\beta$ & Std. Error & Percentage Change & $p$-Value & $H_0$ Rejected \\
\midrule
Format Parquet & 7.2501 &  & +0.0\% &  &  \\
Format Delta & 0.2512 & 0.006643 & +28.6\% & 0.0000 & True \\
Format Hudi & 0.0786 & 0.006636 & +8.2\% & 0.0000 & True \\
Format Iceberg & 0.0696 & 0.006660 & +7.2\% & 0.0000 & True \\
\bottomrule
\end{tabular}
\caption{TPC-DS 10GB, Elapsed Time}
\end{table}

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Fact

In [23]:
for format in ['delta', 'hudi', 'iceberg']:

    for target_metric in target_metrics:
        
        for data_set in data_sets:

            df_rq2 = master_df[
                (master_df['format'] == format) &
                (master_df['data_set'] == data_set) 
            ].copy()

            df_rq2['format'] = pd.Categorical(df_rq2['format'], categories=formats)

            y = df_rq2[target_metric].values + EPSILON
            group_data = df_rq2['query'].to_numpy()

            X_df = patsy.dmatrix("C(optimization, Treatment(reference='none')) * C(block_size, Treatment(reference='128MiB'))", data=df_rq2, return_type='dataframe')
            X = X_df.values
            feature_names = X_df.columns.tolist()

            gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
            gp_model.fit(y=y, X=X)
            fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

            summary_data = []
            for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
                z_stat = coef / std_err
                p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
                pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
                
                summary_data.append({
                    'Factor':  'No opt: 128Mib' if name == 'Intercept' else clean_patsy_name(name),
                    'Coefficient $\\beta$': f'{coef:.4f}',
                    'Std. Error': f'{std_err:.4f}' if name != 'Intercept' else '',
                    'Percentage Change': f'{pct_change:+.1f}\\%' if not np.isnan(pct_change) else 'Baseline',
                    '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                    '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
                })

            summary_df = pd.DataFrame(summary_data)
            caption_text = f'Format {format.capitalize()}, TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}'
            print_df_latex(summary_df, caption_text)
            

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Factor & Coefficient $\beta$ & Std. Error & Percentage Change & $p$-Value & $H_0$ Rejected \\
\midrule
No opt: 128Mib & 6.1419 &  & +0.0\% &  &  \\
Bloom & -0.0173 & 0.0069 & -1.7\% & 0.0123 & True \\
Partitioning & 0.1538 & 0.0069 & +16.6\% & 0.0000 & True \\
Zorder & 0.0144 & 0.0055 & +1.5\% & 0.0085 & True \\
256Mib & 0.0065 & 0.0055 & +0.7\% & 0.2344 & False \\
64Mib & -0.0306 & 0.0055 & -3.0\% & 0.0000 & True \\
Bloom: 256Mib & -0.0116 & 0.0101 & -1.1\% & 0.2518 & False \\
Partitioning: 256Mib & -0.0165 & 0.0100 & -1.6\% & 0.0972 & False \\
Zorder: 256Mib & -0.0145 & 0.0091 & -1.4\% & 0.1127 & False \\
Bloom: 64Mib & 0.0304 & 0.0101 & +3.1\% & 0.0025 & True \\
Partitioning: 64Mib & 0.0119 & 0.0100 & +1.2\% & 0.2349 & False \\
Zorder: 64Mib & 0.0184 & 0.0092 & +1.9\% & 0.0439 & True \\
\bottomrule
\end{tabular}
\caption{Format Delta, TPC-DS 1GB, Elapsed Time}
\end{table}

\begin{table}[h!]
\centering
\begin{tabular}{llll

In [24]:
for target_metric in target_metrics:
        
    for data_set in data_sets:

        for optimization in optimizations:

            df_rq3 = master_df[
                ((master_df['format'] == 'parquet') & (master_df['data_set'] == data_set)) |
                ((master_df['format'] != 'parquet') & (master_df['data_set'] == data_set) & (master_df['optimization'] == optimization))
            ].copy()

            # df_rq3['format'] = pd.Categorical(df_rq3['format'], categories=formats)
            df_rq3['format_variation'] = df_rq3['format'] + '_' + df_rq3['optimization'] + '_' + df_rq3['block_size'].astype(str)

            y = df_rq3[target_metric].values + EPSILON
            group_data = df_rq3['query'].to_numpy()

            X_df = patsy.dmatrix("C(format_variation, Treatment(reference='parquet_none_128MiB'))", data=df_rq3, return_type='dataframe')
            X = X_df.values
            feature_names = X_df.columns.tolist()

            gp_model = gpb.GPModel(group_data=group_data, likelihood='gamma')
            gp_model.fit(y=y, X=X)
            fixed_effects, std_errs = gp_model.get_coef(format_pandas=False, std_err=True)

            summary_data = []
            for name, coef, std_err in zip(feature_names, fixed_effects, std_errs):
                z_stat = coef / std_err
                p_value = 2 * scipy_stats.norm.sf(np.abs(z_stat))
                pct_change = (np.exp(coef) - 1) * 100 if name != 'Intercept' else 0
                
                summary_data.append({
                    'Factor': 'Parquet None 128MiB' if name == 'Intercept' else clean_patsy_name(name).replace('_', ' '),
                    'Coef. $\\beta$': f'{coef:.4f}',
                    'Std. Error': f'{std_err:.4f}' if name != 'Intercept' else '',
                    'Pct. Change': f'{pct_change:+.1f}\\%',
                    '$p$-Value': f'{p_value:.4f}' if name != 'Intercept' else '',
                    '$H_0$ Rejected': str(p_value <= 0.05) if name != 'Intercept' else ''
                })

            summary_df = pd.DataFrame(summary_data)
            caption_text=f'Optimization {optimization.capitalize()}, TPC-DS {data_set.split('_')[1]}GB, {re.sub(r'(?<!^)(?=[A-Z])', ' ', target_metric).title()}'
            print_df_latex(summary_df, caption_text)

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Factor & Coef. $\beta$ & Std. Error & Pct. Change & $p$-Value & $H_0$ Rejected \\
\midrule
Parquet None 128MiB & 5.1632 &  & +0.0\% &  &  \\
Delta None 128Mib & 1.2596 & 0.0177 & +252.4\% & 0.0000 & True \\
Delta None 256Mib & 1.2796 & 0.0177 & +259.5\% & 0.0000 & True \\
Delta None 64Mib & 1.2255 & 0.0177 & +240.6\% & 0.0000 & True \\
Hudi None 128Mib & 0.1565 & 0.0169 & +16.9\% & 0.0000 & True \\
Hudi None 256Mib & 0.1563 & 0.0169 & +16.9\% & 0.0000 & True \\
Hudi None 64Mib & 0.0678 & 0.0169 & +7.0\% & 0.0001 & True \\
Iceberg None 128Mib & 0.0235 & 0.0169 & +2.4\% & 0.1624 & False \\
Iceberg None 256Mib & 0.0238 & 0.0121 & +2.4\% & 0.0492 & True \\
Iceberg None 64Mib & -0.0248 & 0.0077 & -2.5\% & 0.0012 & True \\
\bottomrule
\end{tabular}
\caption{Optimization None, TPC-DS 1GB, Elapsed Time}
\end{table}

\begin{table}[h!]
\centering
\begin{tabular}{llllll}
\toprule
Factor & Coef. $\beta$ & Std. Error & Pct. Change & $p$-